In [133]:
from google.colab import files

uploaded = files.upload()

Saving sample_submission.csv to sample_submission (4).csv


In [134]:
from google.colab import files

uploaded = files.upload()

Saving test_fe.csv to test_fe (4).csv
Saving train_fe.csv to train_fe (4).csv


In [135]:
!pip install -q xgboost

In [136]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import seaborn as sns

In [137]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [138]:
train = pd.read_csv("train_fe.csv")
test = pd.read_csv("test_fe.csv")

print(train.shape)
print(test.shape)

(577347, 19)
(247435, 18)


In [139]:
DROP_COLS = [
    "spectral_type",
    "galaxy_population"
]

train.drop(
    columns=DROP_COLS,
    inplace=True
)

test.drop(
    columns=DROP_COLS,
    inplace=True
)

In [140]:
for df in [train, test]:

    df["alpha_rad"] = np.radians(df["alpha"])
    df["delta_rad"] = np.radians(df["delta"])

    df["x_coord"] = (
        np.cos(df["delta_rad"])
        * np.cos(df["alpha_rad"])
    )

    df["y_coord"] = (
        np.cos(df["delta_rad"])
        * np.sin(df["alpha_rad"])
    )

    df["z_coord"] = np.sin(
        df["delta_rad"]
    )

    df["u_div_g"] = (
        df["u"] /
        (df["g"] + 1e-6)
    )

    df["g_div_r"] = (
        df["g"] /
        (df["r"] + 1e-6)
    )

    df["r_div_i"] = (
        df["r"] /
        (df["i"] + 1e-6)
    )

    df["i_div_z"] = (
        df["i"] /
        (df["z"] + 1e-6)
    )

    df["redshift_sq"] = (
        df["redshift"] ** 2
    )

    df["redshift_cube"] = (
        df["redshift"] ** 3
    )

In [141]:
for df in [train, test]:

    df["sin_alpha"] = np.sin(
        np.radians(df["alpha"])
    )

    df["cos_alpha"] = np.cos(
        np.radians(df["alpha"])
    )

    df["sin_delta"] = np.sin(
        np.radians(df["delta"])
    )

    df["cos_delta"] = np.cos(
        np.radians(df["delta"])
    )

    df["redshift_griz"] = (
        df["redshift"]
        * df["griz"]
    )

    df["redshift_ugri"] = (
        df["redshift"]
        * df["ugri"]
    )

    df["redshift_u_g"] = (
        df["redshift"]
        * df["u_g"]
    )

    df["redshift_g_r"] = (
        df["redshift"]
        * df["g_r"]
    )

    df["redshift_r_i"] = (
        df["redshift"]
        * df["r_i"]
    )

    df["redshift_sq_griz"] = (
        df["redshift_sq"]
        * df["griz"]
    )

    df["alpha_delta"] = (
        df["alpha"]
        * df["delta"]
    )

    df["alpha_redshift"] = (
        df["alpha"]
        * df["redshift"]
    )

    df["delta_redshift"] = (
        df["delta"]
        * df["redshift"]
    )

In [142]:
for df in [train, test]:

    df["redshift_fourth"] = (
        df["redshift"] ** 4
    )

    df["redshift_griz_sq"] = (
        df["redshift_sq"]
        * df["griz"]
    )

    df["redshift_ugri_sq"] = (
        df["redshift_sq"]
        * df["ugri"]
    )

    df["redshift_cube_griz"] = (
        df["redshift_cube"]
        * df["griz"]
    )

    df["redshift_cube_ugri"] = (
        df["redshift_cube"]
        * df["ugri"]
    )

    df["redshift_log_griz"] = (
        np.log1p(df["redshift"])
        * df["griz"]
    )

    df["redshift_log_ugri"] = (
        np.log1p(df["redshift"])
        * df["ugri"]
    )

In [143]:
all_redshift = pd.concat([
    train["redshift"],
    test["redshift"]
])

bins = pd.qcut(
    all_redshift,
    q=30,
    duplicates="drop",
    retbins=True
)[1]

for df in [train, test]:

    df["redshift_bin"] = pd.cut(
        df["redshift"],
        bins=bins,
        labels=False,
        include_lowest=True
    )

In [144]:
TARGET = "class"

X = train.drop(
    columns=[
        "id",
        TARGET
    ]
)

X_test = test.drop(
    columns=["id"]
)

target_encoder = LabelEncoder()

y = target_encoder.fit_transform(
    train[TARGET]
)

print(X.shape)

(577347, 47)


In [145]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = []

test_probs = np.zeros(
    (
        len(test),
        3
    )
)

In [146]:
for fold, (
    train_idx,
    valid_idx
) in enumerate(
    skf.split(X, y)
):

    print(
        f"\nFold {fold+1}"
    )

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y[train_idx]
    y_valid = y[valid_idx]

    model = XGBClassifier(

        objective="multi:softprob",

        num_class=3,

        n_estimators=2500,

        learning_rate=0.03,

        max_depth=8,

        subsample=0.8,

        colsample_bytree=0.8,

        min_child_weight=3,

        reg_alpha=0.1,

        reg_lambda=1.0,

        random_state=42,

        tree_method="hist",

        device="cuda",

        eval_metric="mlogloss"
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_valid
    )

    score = balanced_accuracy_score(
        y_valid,
        pred
    )

    cv_scores.append(score)

    print(score)

    test_probs += (
        model.predict_proba(
            X_test
        )
        / skf.n_splits
    )


Fold 1
0.9569585917341775

Fold 2
0.95919005833421

Fold 3
0.9578255312778118

Fold 4
0.9578889271734857

Fold 5
0.9585976405460089


In [147]:
print(
    "Mean CV:",
    np.mean(cv_scores)
)

print(
    "Std CV:",
    np.std(cv_scores)
)

Mean CV: 0.9580921498131388
Std CV: 0.00075607708663657


In [148]:
feature_importance = pd.DataFrame({

    "feature":
        X.columns,

    "importance":
        model.feature_importances_

})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

feature_importance.head(40)

,feature,importance
14,redshift_log,0.190101
7,redshift,0.151993
13,griz,0.100433
41,redshift_ugri_sq,0.094705
43,redshift_cube_ugri,0.082982
24,redshift_sq,0.069924
12,ugri,0.049559
39,redshift_fourth,0.031429
3,g,0.016341
46,redshift_bin,0.014818


In [149]:
feature_importance.to_csv(
    "xgb_v3_importance.csv",
    index=False
)

In [150]:
pd.DataFrame({

    "fold":
        range(
            1,
            len(cv_scores)+1
        ),

    "score":
        cv_scores

}).to_csv(
    "xgb_v3_cv_scores.csv",
    index=False
)

In [151]:
preds = np.argmax(
    test_probs,
    axis=1
)

submission = pd.DataFrame({

    "id":
        test["id"],

    "class":
        target_encoder.inverse_transform(
            preds
        )
})

submission.to_csv(
    "submission_xgb_v3.csv",
    index=False
)

In [152]:
prob_df = pd.DataFrame(
    test_probs,
    columns=target_encoder.classes_
)

prob_df.insert(
    0,
    "id",
    test["id"]
)

prob_df.to_csv(
    "xgb_v3_prob.csv",
    index=False
)

In [153]:
from google.colab import files

files.download(
    "submission_xgb_v3.csv"
)

files.download(
    "xgb_v3_prob.csv"
)

files.download(
    "xgb_v3_importance.csv"
)

files.download(
    "xgb_v3_cv_scores.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>